In [ ]:
# print(123)

In [ ]:
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from tqdm.auto import tqdm
from minsearch import VectorSearch, Index
import numpy as np

## Q1: Embedding a query

In [ ]:
embed = Embedder()
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)
v_query.shape

In [ ]:
v_query[0]

## Q2: Cosine similarity

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
filename="02-vector-search/lessons/07-sqlitesearch-vector.md"
q2_content = None
for i in documents:
   if i["filename"] == filename:
      print("Found it")
      print(f"{i['filename']}")
      q2_content = i['content']

In [ ]:
v2 = embed.encode(q2_content)
v2.shape

In [ ]:
score = v2.dot(v_query)
score

## Q3: Chunking and search by hand

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
texts = [chunk['content'] for chunk in chunks]

batch_size = 50
X = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

In [ ]:
scores = X.dot(query)
idx = np.argmax(scores)

filename_q3 = chunks[idx]['filename']
score_q3 = round(scores[idx],2)
print(f'Filename is "{filename_q3}" with a score: {score_q3}.')

## Q4: Vector search with minsearch

In [ ]:
vindex = VectorSearch()
vindex.fit(X, chunks)

In [ ]:
query_q4 = "What metric do we use to evaluate a search engine?"
query_vector_q2 = embed.encode(query_q4)

result_q4 = vindex.search(query_vector_q2, num_results=1)[0]
filename_q4 = result_q4["filename"]

In [ ]:
print(f'Filename is "{filename_q4}".')

## Q5: Text search vs vector search

In [ ]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [ ]:
query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embed.encode(query_q5)

In [ ]:
top5_text = text_index.search(query_q5, num_results=5)
top5_text_filenames = [item["filename"] for item in top5_text]

top5_text_filenames

In [ ]:
scores_q5 = X.dot(v_q5)
top5_idx = np.argsort(scores_q5)[-5:]
top5_idx = top5_idx[::-1]

top5_vector = [chunks[i] for i in top5_idx]
top5_vector_scores = scores_q5[top5_idx]

In [ ]:
top5_vector_filenames = [item["filename"] for item in top5_vector]

list(zip(top5_vector_scores, top5_vector_filenames))

In [ ]:
comparison = {"text": top5_text_filenames, "vector": top5_vector_filenames}
vector_only = sorted(set(top5_vector_filenames) - set(top5_text_filenames))
comparison, vector_only

## Q6: Hybrid search

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query_q6 = "How do I give the model access to tools?"
v_q6 = embed.encode(query_q6)

vector_results = vector_index.search(v_q6, num_results=5)
text_results = text_index.search(query_q6, num_results=5)

results = rrf([vector_results, text_results])
results[0]["filename"]